In [5]:
from lib.data_loader import DataLoader
from lib.feature_engineering import *
import pandas as pd
from lib.model_inference import model_inference



# if __name__ == "__main__": 


SENSOR_COLS = [
    'Acceleration X (g)',
    'Acceleration Y (g)',
    'Acceleration Z (g)',
    'AE (V)'
]

CONTROLLER_COLS = [
    'feedrate', 'mainSpndLoad', 'mainSpndSpd',
    'X_load', 'Y_load', 'Z_load'
]

variance_zero_cols = ['toolSpndLoad', 'toolSpndSpd', 'toolSpndStatus', 'A_load', 'progName', 'progStatus', 'mainSpndStatus', 'coolStatus', 'operMode']

evalset_list = [1]
cut_list = list(range(2, 27))
CTRL_DROP_COLS = ['timestamp']


# Feature extraction 
records = []
skipped = []

loader = DataLoader("tcdata/Controller_Data", "tcdata/Sensor_Data")

for set_no in evalset_list:
    for cut_no in cut_list:
        print(f"  evalset_{set_no:02d} | Cut {cut_no:02d} ...", end=' ')

        sensor_df = loader.get_sensor_data(set_no, cut_no)
        ctrl_df   = loader.get_controller_data(set_no, cut_no)
        


        if sensor_df.empty or ctrl_df.empty:
            print("⚠ skipped (no data)")
            skipped.append((set_no, cut_no))
            continue
        ctrl_df.drop(columns = variance_zero_cols, inplace = True) 
        ctrl_df.drop(columns = CTRL_DROP_COLS, inplace = True )
        row = {'set_no': set_no, 'cut_no': cut_no}

        # ── Sensor features ──────────────────────────────────────────
        for col in SENSOR_COLS:
            if col not in sensor_df.columns:
                continue
            signal = sensor_df[col].dropna().to_numpy(dtype=float, copy=True)  # ← safest
            col_name = safe_col_name(col)
            row.update(sensor_features(signal, col_name))

        # ── Controller features ──────────────────────────────────────
        row.update(controller_features(ctrl_df))

        records.append(row)
        if len(records) == 1:
            print(f"\n── Feature columns ({len(row)} total) ──")
            # for col in row.keys():
            #     print(f"  {col}")
        print(f"✅  ({len(sensor_df):,} sensor rows)")

features_test = pd.DataFrame(records)

print(features_test.columns)

nan_cols = ['B_load_trend_r2']
features_test.drop(columns = nan_cols, inplace = True) 


    
    


    

  evalset_01 | Cut 02 ... 
── Feature columns (318 total) ──
✅  (1,396,217 sensor rows)
  evalset_01 | Cut 03 ... ✅  (1,396,797 sensor rows)
  evalset_01 | Cut 04 ... ✅  (1,332,802 sensor rows)
  evalset_01 | Cut 05 ... ✅  (1,399,237 sensor rows)
  evalset_01 | Cut 06 ... ✅  (1,327,272 sensor rows)
  evalset_01 | Cut 07 ... ✅  (1,468,069 sensor rows)
  evalset_01 | Cut 08 ... ✅  (1,329,449 sensor rows)
  evalset_01 | Cut 09 ... ✅  (1,331,721 sensor rows)
  evalset_01 | Cut 10 ... ✅  (1,328,633 sensor rows)
  evalset_01 | Cut 11 ... ✅  (1,331,755 sensor rows)
  evalset_01 | Cut 12 ... ✅  (1,465,810 sensor rows)
  evalset_01 | Cut 13 ... ✅  (1,396,060 sensor rows)
  evalset_01 | Cut 14 ... ✅  (1,328,179 sensor rows)
  evalset_01 | Cut 15 ... ✅  (1,397,282 sensor rows)
  evalset_01 | Cut 16 ... ✅  (1,329,101 sensor rows)
  evalset_01 | Cut 17 ... ✅  (1,532,264 sensor rows)
  evalset_01 | Cut 18 ... ✅  (1,329,995 sensor rows)
  evalset_01 | Cut 19 ... ✅  (1,397,760 sensor rows)
  evalset_0

In [22]:
# import joblib
# import numpy as np

import joblib 
import numpy as np

# Load the trained ElasticNet model
enet = joblib.load('model/enet_model.pkl')
print("Model loaded from enet_model.pkl")

# Load the scaler
scaler = joblib.load('model/scaler.pkl')
print("Scaler loaded from scaler.pkl")

# Load the feature indices
top_indices = np.load('model/top_indices.npy')
print("Feature indices loaded from top_indices.npy")


def model_inference(input_features):
    """Generate wear estimate from preprocessed features.
    """
    # apply same feature selection
    print(input_features.shape)
    X_scaled = scaler.transform(input_features.reshape(1,-1))
    X_scaled = X_scaled[:, top_indices]
    
    # predict
    pred = enet.predict(X_scaled)[0]
    return float(pred)


Model loaded from enet_model.pkl
Scaler loaded from scaler.pkl
Feature indices loaded from top_indices.npy


In [ ]:
# features_test.insert(0, 'cut_no', output_csv['cut_num'])

In [30]:
features_test.columns

Index(['cut_no', 'Acceleration_X_g_min', 'Acceleration_X_g_max',
       'Acceleration_X_g_mean', 'Acceleration_X_g_std',
       'Acceleration_X_g_skew', 'Acceleration_X_g_kurtosis',
       'Acceleration_X_g_energy', 'Acceleration_X_g_entropy',
       'Acceleration_X_g_zero_crossings',
       ...
       'Z_load_trend_r2', 'B_load_mean', 'B_load_std', 'B_load_rms',
       'B_load_max', 'B_load_min', 'B_load_range', 'B_load_iqr',
       'B_load_slope', 'cut_duration_s'],
      dtype='str', length=316)

In [31]:

# output_csv['set_num'] = output_csv['set_num'].apply(lambda x: f'evalset_{int(x):02d}')



# features_test.drop(columns=['set_no', 'cut_no'], inplace=True)



print(features_test.columns)

predictions = []
for i, row in features_test.iterrows():
    pred = model_inference(row.to_numpy())
    predictions.append(pred)

output_csv['pred'] = predictions
output_csv.to_csv('result.csv', index=False)


Index(['cut_no', 'Acceleration_X_g_min', 'Acceleration_X_g_max',
       'Acceleration_X_g_mean', 'Acceleration_X_g_std',
       'Acceleration_X_g_skew', 'Acceleration_X_g_kurtosis',
       'Acceleration_X_g_energy', 'Acceleration_X_g_entropy',
       'Acceleration_X_g_zero_crossings',
       ...
       'Z_load_trend_r2', 'B_load_mean', 'B_load_std', 'B_load_rms',
       'B_load_max', 'B_load_min', 'B_load_range', 'B_load_iqr',
       'B_load_slope', 'cut_duration_s'],
      dtype='str', length=316)
(316,)
(316,)
(316,)
(316,)
(316,)
(316,)
(316,)
(316,)
(316,)
(316,)
(316,)
(316,)
(316,)
(316,)
(316,)
(316,)
(316,)
(316,)
(316,)
(316,)
(316,)
(316,)
(316,)
(316,)
(316,)
